# Neo4j Aura BC Private Link Connectivity Test

Validates connectivity from Databricks serverless compute to Neo4j Aura Business Critical
over Azure Private Link via the Application Gateway v2 L4 TCP proxy. Traffic flows:

```
Databricks Serverless --> NCC Private Endpoint --> Private Link tunnel
    --> Application Gateway v2 (L4 TCP passthrough, port 7687) --> Neo4j Aura BC
```

**Prerequisites:**
- Azure infrastructure deployed (`setup_azure.py phase1` then `phase2`)
- App Gateway public IP added to Aura BC allowlist (`manage_ip_allowlist.py add`)
- NCC created and private endpoint rule added (`deploy.py create-ncc`, `create-pe-rule`)
- Connection approved (`deploy.py approve`) and NCC attached to this workspace (`attach-ncc`)
- Secrets stored in a Databricks secret scope (`deploy.py setup-secrets`)

**Key constraints:**
- Must use `bolt+s://` (not `neo4j+s://`) — routing table discovery bypasses the proxy
- Driver must set `max_connection_lifetime < 300` and `liveness_check_timeout < 300` due to Azure Private Link ~5 min idle timeout

In [ ]:
%pip install neo4j>=5.20.0

---

## Setup: Create Secret Scope

Run this once from your terminal to store Neo4j credentials:

```bash
uv run python deploy.py setup-secrets --profile <your-profile>
```

Or manually via the Databricks CLI:

```bash
databricks secrets create-scope neo4j-appgw-poc
databricks secrets put-secret neo4j-appgw-poc password --string-value '<NEO4J_PASSWORD>'
```

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# The NCC private endpoint rule domain must be the real Aura FQDN so that the
# TLS SNI hostname matches what Aura BC expects. Using a made-up private domain
# causes Aura to reject the TLS handshake.
NEO4J_DOMAIN = "xxxxxxxx.databases.neo4j.io"  # <-- Replace with your Aura FQDN

# bolt+s:// is required for Aura BC through the Application Gateway.
# neo4j+s:// performs routing table discovery which bypasses the proxy entirely.
NEO4J_URI = f"bolt+s://{NEO4J_DOMAIN}:7687"

# Credentials from Databricks secrets
SCOPE_NAME = "neo4j-appgw-poc"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = dbutils.secrets.get(SCOPE_NAME, "password")

# Keepalive settings for Azure Private Link ~5 min idle timeout
MAX_CONNECTION_LIFETIME = 240  # seconds, must be < 300
LIVENESS_CHECK_TIMEOUT = 120  # seconds, must be < 300
CONNECTION_ACQUISITION_TIMEOUT = 30  # seconds

print("Configuration loaded:")
print(f"  Domain: {NEO4J_DOMAIN}")
print(f"  URI:    {NEO4J_URI}")
print(f"  User:   {NEO4J_USER}")
print(f"  Max connection lifetime: {MAX_CONNECTION_LIFETIME}s")
print(f"  Liveness check timeout:  {LIVENESS_CHECK_TIMEOUT}s")

---

## Test 1: TCP Connectivity

Verifies the network path is open from serverless compute to the Application Gateway
over Private Link on port 7687 (Bolt).

In [ ]:
import subprocess
import time

print("=" * 60)
print("TEST: TCP Connectivity over Private Link")
print("=" * 60)
print(f"\nTarget: {NEO4J_DOMAIN}:7687")

try:
    start = time.time()
    result = subprocess.run(
        ["nc", "-zv", NEO4J_DOMAIN, "7687"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10,
    )
    elapsed = (time.time() - start) * 1000
    output = (result.stdout.decode() + result.stderr.decode()).strip()

    if result.returncode == 0:
        print(f"\n[PASS] TCP connection established in {elapsed:.1f}ms")
        if output:
            print(f"  Raw: {output}")
    else:
        print(f"\n[FAIL] Cannot reach {NEO4J_DOMAIN}:7687")
        print(f"  Output: {output}")
        print("\n  Check that:")
        print("  - NCC private endpoint status is ESTABLISHED")
        print("  - NCC is attached to this workspace")
        print("  - Domain in NCC rule matches NEO4J_DOMAIN above")

except Exception as e:
    print(f"\n[FAIL] Error: {e}")

---

## Test 2: Neo4j Driver Connectivity

Connects using the Neo4j Python driver over `bolt+s://` with keepalive settings
tuned for the Azure Private Link idle timeout (~300s). Authenticates and runs a test query.

In [ ]:
from neo4j import GraphDatabase
import time

print("=" * 60)
print("TEST: Neo4j Driver over Private Link")
print("=" * 60)
print(f"\nURI: {NEO4J_URI}")

try:
    start = time.time()
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
        max_connection_lifetime=MAX_CONNECTION_LIFETIME,
        liveness_check_timeout=LIVENESS_CHECK_TIMEOUT,
        connection_acquisition_timeout=CONNECTION_ACQUISITION_TIMEOUT,
    )
    driver.verify_connectivity()
    connect_ms = (time.time() - start) * 1000
    print(f"\n[PASS] Connected and authenticated in {connect_ms:.1f}ms")

    # Test query
    records, _, _ = driver.execute_query(
        "RETURN 'Connected over Private Link via bolt+s' AS message"
    )
    print(f"[PASS] Query result: {records[0]['message']}")

    # Server info
    records, _, _ = driver.execute_query(
        "CALL dbms.components() YIELD name, versions RETURN name, versions"
    )
    for record in records:
        print(f"\n  Server: {record['name']} {record['versions']}")

    driver.close()
    total_ms = (time.time() - start) * 1000

    print(f"\n  Connection: {connect_ms:.1f}ms")
    print(f"  Total:      {total_ms:.1f}ms")
    print("\n" + "=" * 60)
    print("PRIVATE LINK CONNECTIVITY VERIFIED")
    print("=" * 60)

except Exception as e:
    print(f"\n[FAIL] {e}")
    print("\n  Check that:")
    print("  - App Gateway public IP is in Aura BC allowlist (manage_ip_allowlist.py list)")
    print("  - Password in secret scope is correct")
    print("  - URI uses bolt+s:// (not neo4j+s://)")
    print("  - App Gateway L4 listener is active on port 7687 (setup_azure.py status)")